# Ordered Logistic Regression Results Dataset Exploration with `mlcroissant`
This notebook provides a walk-through for loading and exploring the FAIR^2 dataset, based on ordered logistic regression outputs for adoption predictors of indigenous and modern knowledge in rangeland management practices in Northern Kenya, using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant Schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Metadata access
metadata = dataset.metadata
print(f"Dataset Title: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Published: {metadata.datePublished}")
print(f"Version: {metadata.version}")
print(f"License: {metadata.license}")

## 2. Data Overview
Review available record sets, their `@id`s, and contained fields. We'll explicitly print the available record sets and their columns/fields for inspection.

In [ ]:
# List all record sets by @id and name
record_sets = dataset.record_sets
if not record_sets:
    print("No record sets found in the Croissant metadata.")
else:
    print("Available Record Sets:")
    for rs in record_sets:
        print(f"@id: {rs['@id']}, name: {rs.get('name', '(no name)')}")

    # For each record set, list available fields and columns by their @id
    print("\nFields/Columns for each Record Set:")
    for rs in record_sets:
        fields = rs.get('field', [])
        if isinstance(fields, dict): fields = [fields]
        columns = rs.get('column', [])
        if isinstance(columns, dict): columns = [columns]
        if fields:
            for field in fields:
                print(f"  RecordSet {rs['@id']} field @id: {field.get('@id', str(field))}, name: {field.get('name', '(no name)')}")
        if columns:
            for column in columns:
                print(f"  RecordSet {rs['@id']} column @id: {column.get('@id', str(column))}, name: {column.get('name', '(no name)')}")

## 3. Data Extraction
Load data from each record set into a DataFrame. All references are by `@id` per the Croissant/FAIR^2 schema. If there are multiple record sets, we extract from each and inspect available fields.

In [ ]:
# Collect DataFrames from each record set using their @id
dataframes = {}
all_record_set_ids = [rs['@id'] for rs in dataset.record_sets]

for record_set_id in all_record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for record set @id: {record_set_id}")
        print(f"Columns: {df.columns.tolist()}")
        print(df.head(2))
    except Exception as e:
        print(f"Could not load record set {record_set_id}: {e}")

if not dataframes:
    print("No dataframes loaded. Check record set definitions and schema contents.")

## 4. Exploratory Data Analysis (EDA)
Apply data processing: filtering, normalization, grouping, etc. For illustration, we will use the first available record set and attempt numeric field analysis (e.g., log likelihood or coefficients if found among columns).

You may need to adapt the field names below to match those found in Section 3 above.

In [ ]:
# Example EDA on the first record set (adapt field names as appropriate)
import numpy as np

if dataframes:
    # Pick the first record set and its dataframe
    first_rs_id = next(iter(dataframes.keys()))
    df = dataframes[first_rs_id]

    print(f"Analyzing record set @id: {first_rs_id}")
    print(f"Sample records:\n", df.head())

    # Try to find a numeric field for demo purposes
    numeric_columns = df.select_dtypes(include=[np.number]).columns

    if numeric_columns.any():
        numeric_field = numeric_columns[0]
        print(f"Using numeric field: {numeric_field}")
        threshold = df[numeric_field].mean() if not np.isnan(df[numeric_field].mean()) else 0

        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        print(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group by the first non-numeric field (as an example)
        group_fields = df.select_dtypes(exclude=[np.number]).columns
        if len(group_fields) > 0:
            group_field = group_fields[0]
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            print(f"\nGrouped by {group_field}:")
            print(grouped_df.sort_values(numeric_field, ascending=False).head())
    else:
        print("No numeric fields found in this record set.")
else:
    print("No dataframes found for EDA.")

## 5. Visualization
Visualize numeric field distributions and relationships. Example: histogram of a numeric field or boxplot by a categorical field if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and 'numeric_field' in locals() and numeric_field in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # Boxplot by group field if available
    if 'group_field' in locals() and group_field in df.columns:
        plt.figure(figsize=(10,6))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} distribution by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field found for visualization. Adjust field names in previous steps if needed.")

## 6. Conclusion
In this notebook, we've demonstrated how to:
- Load FAIR^2 dataset metadata and record sets using the `mlcroissant` library
- Enumerate available record sets, fields, and columns by `@id`
- Extract and inspect records as pandas DataFrames
- Perform simple EDA: filtering, normalization, and grouping on numeric fields (as available)
- Visualize distributions for numeric variables

For further analysis, explore additional fields using their `@id`s (as printed above), and adapt EDA steps for your use case. Refer to the full Croissant schema for the authoritative listing of all identifiers and relationships.